In [1]:
from dotenv import load_dotenv
load_dotenv()

import uuid
from typing import List
from pydantic import BaseModel, Field

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage
from langchain_core.runnables import RunnableConfig

from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore

In [2]:
store = InMemoryStore()

memory_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")
chat_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash")


class MemoryItem(BaseModel):
    text: str = Field(description="Atomic user memory as a short sentence")
    is_new: bool = Field(description="True if NEW and should be stored. False if duplicate/already known.")


class MemoryDecision(BaseModel):
    should_write: bool = Field(description="Whether to store any memories")
    memories: List[MemoryItem] = Field(default_factory=list, description="Extracted user memories")


memory_extractor = memory_llm.with_structured_output(MemoryDecision)

MEMORY_PROMPT = """You are responsible for updating and maintaining accurate user memory.

CURRENT USER DETAILS (existing memories):
{user_details_content}

TASK:
- Review the user's latest message.
- Extract user-specific info worth storing long-term (identity, stable preferences, ongoing projects/goals).
- For each extracted item, set is_new=true ONLY if it adds NEW information compared to CURRENT USER DETAILS.
- If it is basically the same meaning as something already present, set is_new=false.
- Keep each memory as a short atomic sentence.
- No speculation; only facts stated by the user.
- If there is nothing memory-worthy, return should_write=false and an empty list.
"""

SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant with memory capabilities.
If user-specific memory is available, use it to personalize your responses.

Always personalize when possible by:
    - Addressing the user by name (e.g., "Sure, Aayush...") when known
    - Referencing known projects, tools, or preferences
    - Keeping a friendly, natural tone aimed directly at the user

Only use known user details; do not assume anything.
At the end, suggest 3 relevant follow-up questions based on the current response and the user profile.

The user's memory (which may be empty) is provided as:
{user_details_content}
"""

In [3]:
def remember_node(state: MessagesState, config: RunnableConfig, store: BaseStore):
    user_id = config["configurable"]["user_id"]
    namespace = ("users", user_id, "details")

    existing_items = store.search(namespace)
    existing = "\n".join(f"- {it.value['data']}" for it in existing_items) or "(empty)"

    last_msg = state["messages"][-1].content

    decision: MemoryDecision = memory_extractor.invoke([
        SystemMessage(content=MEMORY_PROMPT.format(user_details_content=existing)),
        {"role": "user", "content": last_msg},
    ])

    if decision.should_write:
        for mem in decision.memories:
            if mem.is_new:
                store.put(namespace, str(uuid.uuid4()), {"data": mem.text})

    return {}   # remember node changes no messages